# Baseline — medical concept extraction (Vietnamese clinical notes)

GLiNER zero-shot NER + ConText rule assertions + SapBERT/FAISS linking. No training, runs on a free Colab **T4**.

See `docs/02_baseline.md` for the walkthrough.

## 1. Setup

Clone the `course` branch and install. Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

In [1]:
!git clone https://github.com/duongtruongbinh/viettel_ai_race_task2 medextract
%cd medextract
!pip -q install -r requirements.txt
!pip -q install -e .

fatal: destination path 'medextract' already exists and is not an empty directory.
/home/binhdt/viettel-ai/notebooks/medextract


## 2. Knowledge bases

The linking step needs the RxNorm + ICD-10 knowledge bases. Their source files are license-gated, so download them yourself (see `INSTALL.md`) and place them in `data/kb/raw/`, then build the indexes.

> If you already have prebuilt `data/kb/processed/*.parquet` + `*.faiss` (e.g. from Google Drive), copy them into `data/kb/processed/` and skip the build cell.

In [6]:
# after placing the raw sources in data/kb/raw/ :
!python -m medextract.kb.build_rxnorm
!python -m medextract.kb.build_icd
!python -m medextract.kb.index --device auto

rxnorm_terms: 107,061 rows, 107,061 rxcui -> data/kb/processed/rxnorm_terms.parquet
tty
SCD     40027
SCDC    28326
SBD     24060
IN      14648
  308135: ['amlodipine 10 MG Oral Tablet']
  243670: ['aspirin 81 MG Oral Tablet']
  866436: ['24 HR metoprolol succinate 50 MG Extended Release Oral Tablet']
  392085: ['guaifenesin 800 MG Oral Tablet']
  313782: ['acetaminophen 325 MG Oral Tablet']
  904475: ['pravastatin sodium 40 MG Oral Tablet']
  197527: ['clonazepam 0.5 MG Oral Tablet']
[icd] using Vietnamese source: data/kb/raw/Phu_luc_Bang_danh_muc_ICD10_FINAL_TT06_2026.xlsx
icd_terms: 15,846 rows, 15,844 codes -> data/kb/processed/icd_terms.parquet
source
QD4469        15844
QD98-COVID        2
  K21.0: ['Bệnh trào ngược dạ dày - thực quản kèm viêm thực quản']
  K21.9: ['Bệnh trào ngược dạ dày - thực quản không kèm viêm thực quản']
  I10: ['Bệnh tăng huyết áp vô căn (nguyên phát)']
  E11.9: ['Bệnh đái tháo đường típ 2, không kèm biến chứng']
  U07.1: ['COVID-19, virus được xác định']


## 3. Run the baseline on the sample notes

In [7]:
!python run.py --config configs/baseline.yaml --input data/sample_input --output out/demo

2026-07-18 15:16:42,468 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:1
2026-07-18 15:16:42,855 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:16:42,882 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-18 15:16:43,167 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:16:43,193 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-18 15:16:43,46

## 4. Look at a prediction

In [8]:
import json
print(json.dumps(json.load(open('out/demo/10.json')), ensure_ascii=False, indent=2))

[
  {
    "text": "đái tháo đường type 2",
    "position": [
      10,
      31
    ],
    "type": "CHẨN_ĐOÁN",
    "assertions": [],
    "candidates": [
      "E11",
      "E11.9"
    ]
  },
  {
    "text": "Than phiền",
    "position": [
      50,
      60
    ],
    "type": "TRIỆU_CHỨNG",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "tê bì hai bàn chân",
    "position": [
      61,
      79
    ],
    "type": "TRIỆU_CHỨNG",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "tiểu nhiều",
    "position": [
      83,
      93
    ],
    "type": "TRIỆU_CHỨNG",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "metformin 500 mg po bid",
    "position": [
      112,
      135
    ],
    "type": "THUỐC",
    "assertions": [],
    "candidates": [
      "861007"
    ]
  },
  {
    "text": "insulin glargine",
    "position": [
      139,
      155
    ],
    "type": "THUỐC",
    "assertions": [],
    "candidates": []
  },
  {
    "text": "9,2 mmo

## 5. Score on the dev set

In [9]:
!python run.py --config configs/baseline.yaml --input data/dev/input --output out/dev_base
!python score.py --pred out/dev_base --gold data/dev

2026-07-18 15:17:03,593 INFO medextract.kb.index: loading SapBERT cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR on cuda:1
2026-07-18 15:17:03,939 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:17:03,967 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/config.json "HTTP/1.1 200 OK"
2026-07-18 15:17:04,253 INFO httpx: HTTP Request: HEAD https://huggingface.co/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-18 15:17:04,282 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cambridgeltl/SapBERT-UMLS-2020AB-all-lang-from-XLMR/47b6bd041ba61311584bb2494edfda5c7d9b719f/tokenizer_config.json "HTTP/1.1 200 OK"
2026-07-18 15:17:04,54